In [ ]:
import os
at_colab = 'COLAB_GPU' in os.environ

if at_colab:
    %pip install rasterio
    %pip install mapclassify

In [ ]:
import os
import wget

url = 'https://raw.githubusercontent.com/gromicho/tools/main/jg_folium.py'
filename = 'jg_folium.py'

# Ensure the file is removed before downloading
if os.path.exists(filename):
    os.remove(filename)

# Download the file (this will now always download the latest version)
wget.download(url, filename)

In [ ]:
import jg_folium as jg

In [ ]:
# Standard library imports
import io
import zipfile

# Third-party imports
import matplotlib.pyplot as plt # for graphics
import requests  # For downloading files
import rasterio  # For reading GeoTIFF files
import pandas as pd  # For handling tabular data
import numpy as np  # For numerical operations

In [ ]:
# url of the zip file
url = 'https://data.worldpop.org/GIS/Population_Density/Global_2000_2020_1km_UNadj/2020/NLD/nld_pd_2020_1km_UNadj_ASCII_XYZ.zip'

# Fetch the ZIP file
response = requests.get(url)
response.raise_for_status()  # Ensure we notice any download issues

# Open the ZIP file in memory
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    # Extract the first file (assuming only one CSV in the ZIP)
    file_name = z.namelist()[0]

    # Read the CSV into a Pandas DataFrame
    with z.open(file_name) as f:
        df = pd.read_csv(f)

In [ ]:
total_Z = df['Z'].sum()
print(f'{total_Z:,.2f}')

In [ ]:
import pandas as pd
import numpy as np


def add_headcount_from_density(
    df: pd.DataFrame,
    x_col: str = 'X',
    y_col: str = 'Y',
    z_col: str = 'Z'
) -> pd.DataFrame:
    '''
    Computes the estimated headcount per grid cell from population density.

    Args:
        df (pd.DataFrame): DataFrame containing longitude, latitude, and population density.
        x_col (str, optional): Column name for longitude. Defaults to 'X'.
        y_col (str, optional): Column name for latitude. Defaults to 'Y'.
        z_col (str, optional): Column name for population density. Defaults to 'Z'.

    Returns:
        pd.DataFrame: The input DataFrame with an additional 'headcount' column.
    '''
    # Constants
    earth_radius_km = 6371  # Earth's radius in km
    grid_resolution_deg = 1 / 120  # 30 arc-seconds = 1/120 in degrees
    deg_to_rad = np.pi / 180  # Convert degrees to radians

    # Convert latitude to radians for cosine correction
    lat_rad = df[y_col] * deg_to_rad

    # Compute area of each grid cell in km²
    grid_area_km2 = (
        (grid_resolution_deg * deg_to_rad) * earth_radius_km *
        (grid_resolution_deg * deg_to_rad) * earth_radius_km *
        np.cos(lat_rad)
    )

    # Compute headcount per grid cell
    df['headcount'] = df[z_col] * grid_area_km2

    return df


In [ ]:
df = add_headcount_from_density( df )
# Sum to get total population
total_population = df['headcount'].sum()
print(f'{total_population:,.2f}')

In [ ]:
# URL of the GeoTIFF file
url = 'https://data.worldpop.org/GIS/Population_Density/Global_2000_2020_1km_UNadj/2020/NLD/nld_pd_2020_1km_UNadj.tif'

# File path to save it locally
local_filename = 'nld_pd_2020_1km_UNadj.tif'

# Download the file
with requests.get(url, stream=True) as response:
    response.raise_for_status()
    with open(local_filename, 'wb') as file:
        for chunk in response.iter_content(chunk_size=8192):
            file.write(chunk)

print(f'Downloaded {local_filename} successfully!')


In [ ]:
def geotiff_to_dataframe(
    file_path: str,
    x_col: str = 'X',
    y_col: str = 'Y',
    z_col: str = 'Z'
) -> pd.DataFrame:
    '''
    Reads a GeoTIFF file and converts it into a Pandas DataFrame
    containing spatial coordinates and population density values.

    Args:
        file_path (str): Path to the GeoTIFF file.
        x_col (str, optional): Column name for longitude. Defaults to 'X'.
        y_col (str, optional): Column name for latitude. Defaults to 'Y'.
        z_col (str, optional): Column name for population density. Defaults to 'Z'.

    Returns:
        pd.DataFrame: A DataFrame containing longitude, latitude, and population density.
    '''
    with rasterio.open(file_path) as src:
        # Read the raster data as an array
        data = src.read(1)  # Read the first (and only) band

        # Get coordinates of non-zero grid cells
        rows, cols = np.where(data > 0)
        xs, ys = src.xy(rows, cols)  # Convert grid indices to lat/lon

    # Create a Pandas DataFrame with configurable column names
    df = pd.DataFrame({x_col: xs, y_col: ys, z_col: data[rows, cols]})

    return df

In [ ]:
df = geotiff_to_dataframe( local_filename )

In [ ]:
total_Z = df['Z'].sum()
print(f'{total_Z:,.2f}')

In [ ]:
f'{add_headcount_from_density(df).headcount.sum():,.2f}'

In [ ]:
def plot_pop(
    X: np.ndarray,
    Y: np.ndarray,
    Z: np.ndarray,
    figsize: tuple = (10, 6),
    marker_size_scale: float = 10000
) -> None:
    '''
    Plots population data as a scatter plot.

    Args:
        X (np.ndarray): Array of X-coordinates (longitude).
        Y (np.ndarray): Array of Y-coordinates (latitude).
        Z (np.ndarray): Array of population counts or densities.
        figsize (tuple, optional): Figure size in inches. Defaults to (10, 6).
        marker_size_scale (float, optional): Scaling factor for marker sizes. Defaults to 10000.

    Returns:
        None: Displays the plot.
    '''
    plt.figure(figsize=figsize)
    plt.scatter(X, Y, marker='.', s=Z / marker_size_scale)
    plt.title(f'{len(X):,} clusters adding to {sum(Z):,.0f}')
    plt.show()


In [ ]:
plot_pop( df.X, df.Y, df.Z )

In [ ]:
df = add_headcount_from_density(df)

In [ ]:
plot_pop( df.X, df.Y, df.headcount )

In [ ]:
import pandas as pd
import geopandas as gpd


def df_to_geodf(
    df: pd.DataFrame,
    x_col: str = 'X',
    y_col: str = 'Y',
    crs: str = 'EPSG:4326'
) -> gpd.GeoDataFrame:
    '''
    Converts a Pandas DataFrame with X (longitude) and Y (latitude) into a GeoDataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame containing coordinate columns.
        x_col (str, optional): Column name for longitude. Defaults to 'X'.
        y_col (str, optional): Column name for latitude. Defaults to 'Y'.
        crs (str, optional): Coordinate Reference System. Defaults to 'EPSG:4326' (WGS84).

    Returns:
        gpd.GeoDataFrame: A GeoDataFrame with Point geometries.
    '''
    # Use geopandas built-in function for efficiency
    geometry = gpd.points_from_xy(df[x_col], df[y_col])

    # Convert to GeoDataFrame
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=crs)

    return gdf


In [ ]:
gdf = df_to_geodf(df)

df.shape,gdf.shape

In [ ]:
# Create the plot
gdf.plot(
    markersize=gdf['headcount'] / 10000,
    alpha=0.7,
    color='black',
    figsize=(13, 8)
)
# Show the plot (optional)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Create the plot
ax = gdf.plot(
    column='headcount', 
    cmap='cividis', 
    alpha=.35, 
    markersize=1, 
    figsize=(13, 8),
    legend=True
)

# Remove x and y axis ticks
ax.set_xticks([])
ax.set_yticks([])

# Remove the frame (optional)
ax.set_frame_on(False)

plt.savefig('wp_nl_population_plot.png', format='png', bbox_inches='tight', pad_inches=0)

# Show the plot
plt.show()


In [ ]:
import contextily as ctx
import geopandas as gpd
import matplotlib.pyplot as plt

# Convert to Web Mercator for basemap compatibility
gdf['geometry'] = gdf.geometry.to_crs(epsg=3857)

# Plot, coloring by population density
ax = gdf.plot(
    column='headcount', 
    cmap='cividis', 
    alpha=.2, 
    markersize=1, 
    legend=True, 
    figsize=(13, 8)
)

# Add basemap
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)

# Remove axis labels
ax.set_xticks([])
ax.set_yticks([])
ax.set_frame_on(False)  # Optional: Remove the border/frame

plt.savefig('wp_nl_ctx_population_plot.png', format='png', bbox_inches='tight', pad_inches=0)

plt.show()


In [ ]:
import folium
from folium.plugins import HeatMap

map_center = [gdf['Y'].mean(), gdf['X'].mean()]
m = folium.Map(location=map_center, zoom_start=7)
heat_data = list(zip(gdf['Y'], gdf['X'], gdf['headcount']))
HeatMap(heat_data, radius=8).add_to(m)
jg.FoliumToPng( m, 'wp_nl_heatmap' )
m  # Show the map

In [ ]:
cbs = gpd.read_parquet('../Administrative/cbs.parquet')

In [ ]:
gdf = gdf.to_crs(cbs.crs)
capelle = cbs[cbs.Municipality.str.contains('Capelle')]
points_in_capelle = gdf[gdf.geometry.within(capelle.geometry.union_all())]
m = capelle.explore(color='green',alpha=.25)
m = points_in_capelle.explore(m=m, color='red')
jg.FoliumToPng( m, 'wp_in_capelle' )
m

In [ ]:
points_in_capelle.shape,points_in_capelle.headcount.sum()

In [ ]:
epsg = 'EPSG:4326' # https://en.wikipedia.org/wiki/World_Geodetic_System#WGS_84

pop = gdf.drop(columns=['Z']).rename(
    columns={'headcount' : 'Population', 'Y' : 'latitude', 'X' : 'longitude'}
).to_crs(epsg)

pop.to_parquet('wp_nl.parquet')

In [ ]:
pop